# Article 5: Multi-Agent Systems Analysis

This notebook analyzes benchmark results from Article 5, comparing multi-agent orchestration patterns across LangGraph, CrewAI, and AutoGen frameworks.

**Key Questions:**
1. Which framework has the highest task completion rate?
2. Which framework is most token-efficient (lowest cost per task)?
3. Which patterns (sequential, parallel, critic, conflict) work best on each framework?
4. What is the collaboration overhead (multi-agent vs single-agent cost)?
5. How does latency scale with agent count and interaction complexity?

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Define distinctive color palette (avoid generic AI aesthetics)
COLORS = {
    'langgraph': '#2E86AB',  # Deep blue
    'crewai': '#A23B72',     # Deep magenta
    'autogen': '#F18F01',    # Warm orange
    'sequential': '#06A77D',  # Teal
    'parallel': '#D62246',    # Crimson
    'critic_refinement': '#4B3F72',  # Deep purple
    'conflict_resolution': '#FFC857'  # Gold
}

## 1. Load Benchmark Results

In [ ]:
# Load results from benchmark run
results_path = Path('../results/data/article_05_benchmarks.json')

if not results_path.exists():
    print(f"⚠️  Results file not found: {results_path}")
    print("Run benchmark first: python benchmarks/run_article_05.py")
else:
    with open(results_path) as f:
        data = json.load(f)
    
    # Convert to DataFrames
    results_df = pd.DataFrame(data['individual_results'])
    summary_df = pd.DataFrame(data['framework_summaries'])
    
    print(f"✓ Loaded {len(results_df)} individual task results")
    print(f"✓ Loaded {len(summary_df)} framework summaries")
    print(f"\nGenerated at: {data['generated_at']}")
    
    # Show first few results
    print("\nSample results:")
    print(results_df[['task_id', 'framework', 'pattern', 'success', 'latency_ms', 'total_tokens']].head())

## 2. Framework Comparison: Completion Rate

In [ ]:
# Plot completion rates by framework
fig, ax = plt.subplots(figsize=(10, 6))

frameworks = summary_df['framework'].tolist()
completion_rates = summary_df['completion_rate'].tolist()

bars = ax.bar(frameworks, [r * 100 for r in completion_rates], 
              color=[COLORS.get(f, '#666') for f in frameworks])

ax.set_ylabel('Completion Rate (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('Framework', fontsize=12, fontweight='bold')
ax.set_title('Multi-Agent Task Completion Rate by Framework', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/charts/article_05/completion_rate_by_framework.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved: completion_rate_by_framework.png")

## 3. Token Efficiency Comparison

In [ ]:
# Plot average tokens per task by framework
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Tokens
avg_tokens = summary_df['avg_tokens'].tolist()
bars1 = ax1.bar(frameworks, avg_tokens,
                color=[COLORS.get(f, '#666') for f in frameworks])

ax1.set_ylabel('Average Tokens per Task', fontsize=12, fontweight='bold')
ax1.set_xlabel('Framework', fontsize=12, fontweight='bold')
ax1.set_title('Token Usage Comparison', fontsize=14, fontweight='bold')

for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.0f}',
            ha='center', va='bottom', fontweight='bold')

# Cost
avg_costs = [c * 1000 for c in summary_df['avg_cost_usd'].tolist()]  # Convert to millicents
bars2 = ax2.bar(frameworks, avg_costs,
                color=[COLORS.get(f, '#666') for f in frameworks])

ax2.set_ylabel('Average Cost per Task ($/1000)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Framework', fontsize=12, fontweight='bold')
ax2.set_title('Cost Comparison', fontsize=14, fontweight='bold')

for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'${height:.2f}',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/charts/article_05/token_cost_efficiency.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved: token_cost_efficiency.png")

## 4. Pattern-Specific Success Rates

In [ ]:
# Extract pattern success rates for each framework
pattern_data = []

for _, row in summary_df.iterrows():
    framework = row['framework']
    for pattern, rate in row['pattern_success_rates'].items():
        pattern_data.append({
            'framework': framework,
            'pattern': pattern,
            'success_rate': rate * 100
        })

pattern_df = pd.DataFrame(pattern_data)

# Pivot for grouped bar chart
pivot_df = pattern_df.pivot(index='pattern', columns='framework', values='success_rate')

# Plot grouped bar chart
fig, ax = plt.subplots(figsize=(12, 6))

pivot_df.plot(kind='bar', ax=ax, color=[COLORS.get(f, '#666') for f in pivot_df.columns])

ax.set_ylabel('Success Rate (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('Orchestration Pattern', fontsize=12, fontweight='bold')
ax.set_title('Pattern-Specific Success Rates by Framework', fontsize=14, fontweight='bold')
ax.set_xticklabels(pivot_df.index, rotation=45, ha='right')
ax.legend(title='Framework', title_fontsize=11, fontsize=10)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('../results/charts/article_05/pattern_success_rates.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved: pattern_success_rates.png")

## 5. Latency Analysis

In [ ]:
# Plot latency distribution by framework
fig, ax = plt.subplots(figsize=(12, 6))

for framework in results_df['framework'].unique():
    framework_data = results_df[results_df['framework'] == framework]
    successful_data = framework_data[framework_data['success'] == True]
    
    if len(successful_data) > 0:
        latencies = successful_data['latency_ms'].tolist()
        ax.hist(latencies, bins=20, alpha=0.6, label=framework, 
                color=COLORS.get(framework, '#666'))

ax.set_xlabel('Latency (ms)', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Task Latency Distribution by Framework', fontsize=14, fontweight='bold')
ax.legend(title='Framework', title_fontsize=11, fontsize=10)

plt.tight_layout()
plt.savefig('../results/charts/article_05/latency_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved: latency_distribution.png")

## 6. Agent Interaction Analysis

In [ ]:
# Plot average agent interactions by pattern
pattern_interactions = results_df[results_df['success'] == True].groupby('pattern')['agent_interactions'].mean()

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(pattern_interactions.index, pattern_interactions.values,
              color=[COLORS.get(p, '#666') for p in pattern_interactions.index])

ax.set_ylabel('Average Agent Interactions', fontsize=12, fontweight='bold')
ax.set_xlabel('Orchestration Pattern', fontsize=12, fontweight='bold')
ax.set_title('Collaboration Complexity by Pattern', fontsize=14, fontweight='bold')
ax.set_xticklabels(pattern_interactions.index, rotation=45, ha='right')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/charts/article_05/agent_interactions.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved: agent_interactions.png")

## 7. Key Insights Summary

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS: Multi-Agent Framework Comparison")
print("="*80)

# Best completion rate
best_completion = summary_df.loc[summary_df['completion_rate'].idxmax()]
print(f"\n1. Highest Completion Rate: {best_completion['framework'].upper()}")
print(f"   → {best_completion['completion_rate']:.1%} of tasks completed successfully")

# Most efficient
best_efficiency = summary_df.loc[summary_df['avg_tokens'].idxmin()]
print(f"\n2. Most Token-Efficient: {best_efficiency['framework'].upper()}")
print(f"   → {best_efficiency['avg_tokens']:.0f} avg tokens/task")
print(f"   → ${best_efficiency['avg_cost_usd']:.4f} avg cost/task")

# Fastest
best_speed = summary_df.loc[summary_df['avg_latency_ms'].idxmin()]
print(f"\n3. Fastest Execution: {best_speed['framework'].upper()}")
print(f"   → {best_speed['avg_latency_ms']:.0f}ms avg latency")

# Pattern strengths
print("\n4. Pattern-Specific Strengths:")
for pattern in pattern_df['pattern'].unique():
    pattern_best = pattern_df[pattern_df['pattern'] == pattern].loc[
        pattern_df[pattern_df['pattern'] == pattern]['success_rate'].idxmax()
    ]
    print(f"   {pattern:25s}: {pattern_best['framework']:10s} ({pattern_best['success_rate']:.0f}%)")

print("\n" + "="*80)

## 8. Mermaid Diagram: Agent Interaction Flows

Generate Mermaid diagrams for each orchestration pattern.

In [ ]:
# Generate Mermaid diagrams for documentation

sequential_diagram = """```mermaid
graph LR
    A[User Query] --> B[Researcher Agent]
    B -->|Research Findings| C[Writer Agent]
    C -->|Draft| D[Critic Agent]
    D -->|Feedback| E{Score >= 7?}
    E -->|No| C
    E -->|Yes| F[Final Output]
```"""

parallel_diagram = """```mermaid
graph TB
    A[User Query] --> B[Orchestrator]
    B --> C[Specialist 1]
    B --> D[Specialist 2]
    B --> E[Specialist 3]
    C --> F[Synthesizer]
    D --> F
    E --> F
    F --> G[Final Output]
```"""

conflict_diagram = """```mermaid
graph TB
    A[User Query] --> B[Agent 1]
    A --> C[Agent 2]
    A --> D[Agent 3]
    B -->|Proposal A| E[Resolver]
    C -->|Proposal B| E
    D -->|Proposal C| E
    E -->|Voting/Supervisor| F[Decision]
```"""

print("Sequential Pipeline:")
print(sequential_diagram)
print("\nParallel Fan-out:")
print(parallel_diagram)
print("\nConflict Resolution:")
print(conflict_diagram)

# Save diagrams to file
diagrams_path = Path('../results/charts/article_05/mermaid_diagrams.md')
diagrams_path.parent.mkdir(parents=True, exist_ok=True)

with open(diagrams_path, 'w') as f:
    f.write("# Multi-Agent Orchestration Patterns\n\n")
    f.write("## Sequential Pipeline\n")
    f.write(sequential_diagram + "\n\n")
    f.write("## Parallel Fan-out\n")
    f.write(parallel_diagram + "\n\n")
    f.write("## Conflict Resolution\n")
    f.write(conflict_diagram + "\n")

print(f"\n✓ Mermaid diagrams saved to: {diagrams_path}")

## Complete

All analysis complete. Charts saved to `results/charts/article_05/`:

1. `completion_rate_by_framework.png`
2. `token_cost_efficiency.png`
3. `pattern_success_rates.png`
4. `latency_distribution.png`
5. `agent_interactions.png`
6. `mermaid_diagrams.md`